---
# **Lab10: PyTorch**
---

# ▶️ GPU tools...

In [6]:
!nvidia-smi

Tue Feb  3 16:19:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:06:00.0 Off |                    0 |
| N/A   33C    P0             43W /  300W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# ✅ Batch Normalization for RGB Images

Define a tensor of shape $[B, 3, H, W]$ filled with random values to simulate a batch of RGB images.

In [7]:
import torch

# Define batch parameters
B, C, H, W = 8, 3, 64, 64
images = torch.rand(B, C, H, W)

print("Batch shape:", images.shape)

Batch shape: torch.Size([8, 3, 64, 64])


## ↘️ TODO...

**Steps to perform channel-wise batch normalization**:

1. Compute the **mean per color channel** by averaging over the batch and spatial dimensions
2. Compute the **standard deviation per color channel** over the same dimensions
3. Inspect the computed statistics to ensure they correspond to the RGB channels
    - print("Mean per channel:", mean)
    - print("Std per channel:", std)


4. Reshape the mean and standard deviation tensors to make them compatible with broadcasting
    - Verify the reshaped dimensions:
        - print("Reshaped mean:", mean.shape)
        - print("Reshaped std:", std.shape)


5. Apply channel-wise centering and normalization using broadcasting

6. Verify the result by recomputing the mean and standard deviation of the normalized batch
7. Check that the normalized images have approximately zero mean and unit variance per channel

    - print("New mean per channel:", new_mean)
    - print("New std per channel:", new_std)



## ➡️ Solution...

In [11]:
# images: [B, 3, H, W]
mean = images.mean(dim=(0, 2, 3))  # shape: [3]
std  = images.std(dim=(0, 2, 3))   # shape: [3]
print("Mean per channel:", mean)

print("Std per channel:", std)

# reshape for broadcasting
mean = mean.view(1, 3, 1, 1)
std  = std.view(1, 3, 1, 1)
print("Reshaped mean:", mean.shape)
print("Reshaped std:", std.shape)

images_norm = (images - mean) / std

# Verify normalization
new_mean = images_norm.mean(dim=(0, 2, 3))
new_std  = images_norm.std(dim=(0, 2, 3))
print("New mean per channel:", new_mean)
print("New std per channel:", new_std)

Mean per channel: tensor([0.4991, 0.5011, 0.5005])
Std per channel: tensor([0.2887, 0.2883, 0.2885])
Reshaped mean: torch.Size([1, 3, 1, 1])
Reshaped std: torch.Size([1, 3, 1, 1])
New mean per channel: tensor([ 7.7765e-08, -7.6485e-08, -1.3097e-07])
New std per channel: tensor([1.0000, 1.0000, 1.0000])


# ✅ Mini multi-head attention block

In [ ]:
# Batch matrix multiplication example
a = torch.randn(2, 4, 5, 4)
b = torch.randn(2, 1, 4, 3)

# Broadcasting happens on the second dimension of b
c = torch.matmul(a, b)  # Equivalent to a@b

print("Shape of c:", c.shape)  # Should be [2, 4, 5, 3]

## ↘️ TODO...

- Implement a **mini multi-head attention block** *from scratch* using only:
    - tensor reshaping (`view`, `reshape`, `transpose`, `permute`)
    - batched matrix multiplication (`matmul`)
    - broadcasting


<br> 🔹 **Setup**

- batch size $B = 4$
- sequence length $T = 16$
- embedding dimension $D = 64$
- number of heads $H = 8$ (so head dimension $d = D/H = 8$)

- Create:
    - input tensor `x` with shape `(B, T, D)`  
    - learnable projection matrices `Wq, Wk, Wv, Wo` with shapes `(D, D)`


<br> 🔹 **Tasks**


- **1) Create the input**
  - Generate `x = torch.randn(B, T, D, requires_grad=True)`
<br>


- **2) Compute Q, K, V**
  - Compute:
    - `Q = x @ Wq`
    - `K = x @ Wk`
    - `V = x @ Wv`
  - Ensure each has shape `(B, T, D)`
<br>

- **3) Split into heads**
  - Reshape and permute so that:
    - `Qh, Kh, Vh` have shape `(B, H, T, d)`
  - Hint: use `.view(B, T, H, d)` then `.transpose(1, 2)`
<br>


- **4) Compute attention scores**
  - Compute scaled dot-product attention:
    $$
    S = \frac{Q_h K_h^T}{\sqrt{d}}
    $$
  - `S` must have shape `(B, H, T, T)`
  - Use `torch.matmul(Qh, Kh.transpose(-2, -1))`
<br>

- **5) Softmax + weighted sum**
  - Apply:
    $$
    A = \mathrm{softmax}(S)
    $$
    $$
    O_h = A V_h
    $$
  - `Oh` must have shape `(B, H, T, d)`
<br>

- **6) Merge heads**
  - Convert `(B, H, T, d)` back to `(B, T, D)` using transpose + reshape.
<br>

- **7) Output projection**
  - Compute final output:
    - `y = out @ Wo`
  - Shape must be `(B, T, D)`

<br> 🔹 What to Submit

- A short printout of shapes at each step:
  - `Qh.shape`, `S.shape`, `A.shape`, `Oh.shape`, `y.shape`
- Confirmation that gradients are computed


<br> 🔹 **Expected Key Shapes**

- `x`: `(B, T, D)`
- `Qh, Kh, Vh`: `(B, H, T, d)`
- `S`: `(B, H, T, T)`
- `A`: `(B, H, T, T)`
- `Oh`: `(B, H, T, d)`
- `y`: `(B, T, D)`

## ➡️ Solution...

In [ ]:
import math
import torch

# -----------------------------
# Final Exercise Solution:
# Mini Multi-Head Attention from scratch
# -----------------------------

torch.manual_seed(0)

# Setup
B, T, D = 4, 16, 64
H = 8
d = D // H
assert D % H == 0

# Input
x = torch.randn(B, T, D, requires_grad=True)

# Learnable projections (D -> D)
Wq = torch.randn(D, D, requires_grad=True)
Wk = torch.randn(D, D, requires_grad=True)
Wv = torch.randn(D, D, requires_grad=True)
Wo = torch.randn(D, D, requires_grad=True)

# 1) Q, K, V: (B, T, D)
Q = x @ Wq
K = x @ Wk
V = x @ Wv
print("Q, K, V:", Q.shape, K.shape, V.shape)

# 2) Split into heads: (B, H, T, d)
# (B, T, D) -> (B, T, H, d) -> (B, H, T, d)
Qh = Q.view(B, T, H, d).transpose(1, 2)
Kh = K.view(B, T, H, d).transpose(1, 2)
Vh = V.view(B, T, H, d).transpose(1, 2)
print("Qh, Kh, Vh:", Qh.shape, Kh.shape, Vh.shape)

# 3) Attention scores: S = Qh @ Kh^T / sqrt(d) -> (B, H, T, T)
S = torch.matmul(Qh, Kh.transpose(-2, -1)) / math.sqrt(d)
print("Scores S:", S.shape)

# # 4) Causal mask (prevent attending to future tokens)
# # mask shape: (T, T), True where we should mask
# mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
# # Broadcast mask to (B, H, T, T) via unsqueeze
# S_masked = S.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))

# # 5) Softmax (numerically stable)
# S_stable = S_masked - S_masked.max(dim=-1, keepdim=True).values
A = torch.softmax(S, dim=-1)
print("Attention weights A:", A.shape)

# 6) Weighted sum: Oh = A @ Vh -> (B, H, T, d)
Oh = torch.matmul(A, Vh)
print("Oh:", Oh.shape)

# 7) Merge heads: (B, H, T, d) -> (B, T, H, d) -> (B, T, D)
out = Oh.transpose(1, 2).contiguous().view(B, T, D)
print("Merged out:", out.shape)

# 8) Output projection: y = out @ Wo -> (B, T, D)
y = out @ Wo
print("y:", y.shape)


In [ ]:
import math
import torch

# -----------------------------
# Final Exercise Solution:
# Mini Multi-Head Attention from scratch
# -----------------------------

torch.manual_seed(0)

# Setup
B, T, D = 4, 16, 64
H = 8
d = D // H
assert D % H == 0

# Input
x = torch.randn(B, T, D, requires_grad=True)

# Learnable projections (D -> D)
Wq = torch.randn(D, D, requires_grad=True)
Wk = torch.randn(D, D, requires_grad=True)
Wv = torch.randn(D, D, requires_grad=True)
Wo = torch.randn(D, D, requires_grad=True)

# 1) Q, K, V: (B, T, D)
Q = x @ Wq
K = x @ Wk
V = x @ Wv
print("Q, K, V:", Q.shape, K.shape, V.shape)

# 2) Split into heads: (B, H, T, d)
# (B, T, D) -> (B, T, H, d) -> (B, H, T, d)
Qh = Q.view(B, T, H, d).transpose(1, 2)
Kh = K.view(B, T, H, d).transpose(1, 2)
Vh = V.view(B, T, H, d).transpose(1, 2)
print("Qh, Kh, Vh:", Qh.shape, Kh.shape, Vh.shape)

# 3) Attention scores: S = Qh @ Kh^T / sqrt(d) -> (B, H, T, T)
S = torch.matmul(Qh, Kh.transpose(-2, -1)) / math.sqrt(d)
print("Scores S:", S.shape)

# # 4) Causal mask (prevent attending to future tokens)
# # mask shape: (T, T), True where we should mask
# mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
# # Broadcast mask to (B, H, T, T) via unsqueeze
# S_masked = S.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))

# # 5) Softmax (numerically stable)
# S_stable = S_masked - S_masked.max(dim=-1, keepdim=True).values
A = torch.softmax(S, dim=-1)
print("Attention weights A:", A.shape)

# 6) Weighted sum: Oh = A @ Vh -> (B, H, T, d)
Oh = torch.matmul(A, Vh)
print("Oh:", Oh.shape)

# 7) Merge heads: (B, H, T, d) -> (B, T, H, d) -> (B, T, D)
out = Oh.transpose(1, 2).contiguous().view(B, T, D)
print("Merged out:", out.shape)

# 8) Output projection: y = out @ Wo -> (B, T, D)
y = out @ Wo
print("y:", y.shape)
